In [ ]:
import gc
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
# ABSA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Comments_ABSA")
# OUTPUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/ABSA_KPI")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# PRODUCT_MASTER_PATH = Path(
#     "/content/drive/MyDrive/Colab Notebooks/Dataset/Feature_KPI_output/product_master.parquet"
# )

In [ ]:
absa_files = sorted(ABSA_DIR.glob("*.parquet"))
print(f"ABSA files: {len(absa_files)}")

dfs = []
for f in tqdm(absa_files, desc="load ABSA"):
    dfs.append(pd.read_parquet(f))
comments = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print("comments:", comments.shape)

In [ ]:
pm = pd.read_parquet(PRODUCT_MASTER_PATH)

# فقط ستون‌های کاربردی برای join / خروجی نهایی
keep_cols = [
    c for c in [
        "id", "title_fa", "Brand", "Category1", "Category2", "sub_category",
        "Price", "price", "Rate", "Rate_cnt",
        "is_view", "is_cart", "is_purchase", "conversion_rate", "drop_off_rate",
        "avg_rate", "comment_count", "avg_like_ratio", "avg_sentiment",
    ] if c in pm.columns
]
products = pm[keep_cols].drop_duplicates(subset=["id"])
del pm
gc.collect()

In [ ]:
aspect_df = (
    comments
    .explode("predicted_aspects")
    .dropna(subset=["predicted_aspects"])
)
print("aspect rows:", aspect_df.shape)


In [ ]:
SENT_COLS = ["positive", "negative", "neutral"]

def sentiment_pivot(df, keys):
    """keys: list of groupby columns → % sentiment + score + total"""
    g = (
        df.groupby(keys + ["predicted_sentiment"])
        .size()
        .unstack(fill_value=0)
    )
    for c in SENT_COLS:
        if c not in g.columns:
            g[c] = 0
    total = g[SENT_COLS].sum(axis=1).replace(0, np.nan)
    out = g[SENT_COLS].div(total, axis=0).mul(100).round(2)
    out["sentiment_score"] = (out["positive"] - out["negative"]).round(2)
    out["total_comments"] = g[SENT_COLS].sum(axis=1).astype(int)
    return out.reset_index()

In [ ]:
product_kpi = sentiment_pivot(comments, ["product_id"])

top_aspect = (
    aspect_df.groupby(["product_id", "predicted_aspects"])
    .size()
    .reset_index(name="cnt")
    .sort_values("cnt", ascending=False)
    .drop_duplicates("product_id")
    [["product_id", "predicted_aspects"]]
    .rename(columns={"predicted_aspects": "top_aspect"})
)

worst_aspect = (
    aspect_df[aspect_df["predicted_sentiment"] == "negative"]
    .groupby(["product_id", "predicted_aspects"])
    .size()
    .reset_index(name="cnt")
    .sort_values("cnt", ascending=False)
    .drop_duplicates("product_id")
    [["product_id", "predicted_aspects"]]
    .rename(columns={"predicted_aspects": "worst_aspect"})
)

product_kpi = (
    product_kpi
    .merge(top_aspect, on="product_id", how="left")
    .merge(worst_aspect, on="product_id", how="left")
)

In [ ]:
comments_meta = comments.merge(
    products[["id", "Brand", "Category1"]],
    left_on="product_id",
    right_on="id",
    how="left",
)

brand_kpi = sentiment_pivot(
    comments_meta.dropna(subset=["Brand"]),
    ["Brand"],
)
category_kpi = sentiment_pivot(
    comments_meta.dropna(subset=["Category1"]),
    ["Category1"],
)

aspect_kpi = sentiment_pivot(aspect_df, ["predicted_aspects"])
aspect_kpi = aspect_kpi.rename(columns={"total_comments": "total_mentions"})

sentiment_distribution = (
    comments["predicted_sentiment"]
    .value_counts()
    .rename_axis("predicted_sentiment")
    .reset_index(name="count")
)

aspect_frequency = (
    aspect_df.groupby("predicted_aspects")
    .size()
    .sort_values(ascending=False)
    .rename_axis("predicted_aspects")
    .reset_index(name="count")
)

product_aspect_kpi = sentiment_pivot(
    aspect_df,
    ["product_id", "predicted_aspects"],
)
product_aspect_kpi = product_aspect_kpi.rename(
    columns={"total_comments": "total_mentions"}
)

In [ ]:
product_master_absa = products.merge(
    product_kpi,
    left_on="id",
    right_on="product_id",
    how="left",
)

# اگر product_id تکراری از merge ماند، پاک کن
if "product_id" in product_master_absa.columns:
    product_master_absa = product_master_absa.drop(columns=["product_id"])


In [ ]:
# granular
product_kpi.to_parquet(OUTPUT_DIR / "product_kpi.parquet", index=False)
brand_kpi.to_parquet(OUTPUT_DIR / "brand_kpi.parquet", index=False)
category_kpi.to_parquet(OUTPUT_DIR / "category_kpi.parquet", index=False)
aspect_kpi.to_parquet(OUTPUT_DIR / "aspect_kpi.parquet", index=False)
product_aspect_kpi.to_parquet(OUTPUT_DIR / "product_aspect_kpi.parquet", index=False)
sentiment_distribution.to_parquet(OUTPUT_DIR / "sentiment_distribution.parquet", index=False)
aspect_frequency.to_parquet(OUTPUT_DIR / "aspect_frequency.parquet", index=False)

# merged golden table
product_master_absa.to_parquet(
    OUTPUT_DIR / "product_master_absa.parquet",
    index=False,
)

print("\n=== SAVED ===")
print("GRANULAR:")
for name in [
    "product_kpi", "brand_kpi", "category_kpi", "aspect_kpi",
    "product_aspect_kpi", "sentiment_distribution", "aspect_frequency",
]:
    p = OUTPUT_DIR / f"{name}.parquet"
    print(f"  {p.name}")

print("MERGED:")
print(f"  product_master_absa.parquet  shape={product_master_absa.shape}")
print("\nDone.")

In [ ]:


# ============================================================
# PATHS  (مسیرها را با محیط خودت عوض کن)
# ============================================================

# اگر product_master نداری، از raw products استفاده کن و ستون‌های id/Brand/Category1 را نگه دار

# ============================================================
# 1) LOAD ABSA RESULTS
# ============================================================

# ستون‌های مورد انتظار: product_id, predicted_sentiment, predicted_aspects, ...

# ============================================================
# 2) LOAD PRODUCT MASTER (behavior + catalog + comment aggregates)
# ============================================================


# ============================================================
# 3) ASPECT EXPLODE (یک‌بار)
# ============================================================


# ============================================================
# 4) HELPER
# ============================================================


# ============================================================
# 5) PRODUCT KPI  (+ top / worst aspect)
# ============================================================


# ============================================================
# 6) BRAND / CATEGORY / ASPECT KPIs
# ============================================================
# برای brand/category باید Brand و Category روی comments باشد


# ============================================================
# 7) جدول طلایی MERGE شده: product_master + ABSA
# ============================================================

# ============================================================
# 8) SAVE
#    A) جداول نرمال (برای PostgreSQL / join آزاد)
#    B) یک جدول طلایی merge‌شده (برای داشبورد و مدل)
# ============================================================
